# Gene Expression Integration & Batch Correction Studio
**Developed by:** Md Fazlay Rabbi and Enamul Haq Rana  
Department of Zoology, University of Rajshahi, Bangladesh  
Contact: rabbi.ms244@gmail.com

Run the code cell below. Missing Python packages are checked and installed automatically when internet access is available.


In [ ]:
# Gene Expression Integration & Batch Correction Studio
# Jupyter-compatible PyQt5 desktop GUI
# Developed by Md Fazlay Rabbi and Enamul Haq Rana
# Department of Zoology, University of Rajshahi, Bangladesh
# Contact: rabbi.ms244@gmail.com
#
# Workflow:
#   1) Load processed GEO microarray CSV files.
#   2) Review per-dataset expression scale and apply selected log2 transformations.
#   3) Integrate common genes and assess batch-associated structure.
#   4) Run ComBat through the standalone pyComBat (combat) package when appropriate.
#   5) Review PCA, boxplots, silhouette score and within-to-between distance ratio.
#   6) Export matrices, metadata, reports, and publication-quality figures.

# ============================================================
# 0. AUTOMATIC DEPENDENCY CHECK / INSTALLATION
# ============================================================
import sys
import subprocess
import importlib

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "PyQt5": "PyQt5",
    "mpmath": "mpmath",
    "patsy": "patsy",
    "combat": "combat",
}


def ensure_packages():
    """Install only missing packages into the current Jupyter/Python environment."""
    missing = []
    for module_name, pip_name in REQUIRED_PACKAGES.items():
        try:
            importlib.import_module(module_name)
        except Exception:
            missing.append((module_name, pip_name))

    if not missing:
        print("All required packages are available.")
        return

    print("Missing package(s) detected:")
    for _, pip_name in missing:
        print(f"  - {pip_name}")

    print("\nInstalling missing package(s) automatically...")
    for module_name, pip_name in missing:
        print(f"Installing {pip_name} ...")
        try:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                pip_name,
            ])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                f"Automatic installation failed for '{pip_name}'.\n\n"
                "The application installs missing packages automatically, but the first installation "
                "requires internet access and permission to install packages in the current Python environment.\n"
                f"Python executable: {sys.executable}"
            ) from exc
        importlib.invalidate_caches()

    failures = []
    for module_name, pip_name in missing:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures.append(f"{pip_name}: {exc}")
    if failures:
        raise RuntimeError(
            "Package installation completed, but one or more packages could not be imported:\n"
            + "\n".join(failures)
            + "\n\nIn Jupyter, restart the kernel once and run this cell again if a newly installed compiled package is not immediately importable."
        )

    print("Package setup completed successfully.\n")


ensure_packages()

# ============================================================
# 1. IMPORTS
# ============================================================
import os
import re
import io
import json
import traceback
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd

# Compatibility shim for standalone pyComBat under NumPy 2.x.
if not hasattr(np, "mat"):
    np.mat = np.asmatrix

from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

from combat.pycombat import pycombat

import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar

from PyQt5.QtCore import Qt, QObject, QThread, pyqtSignal
from PyQt5.QtGui import QFont, QColor
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, QGridLayout,
    QPushButton, QLabel, QFileDialog, QMessageBox, QTabWidget, QTableWidget,
    QTableWidgetItem, QHeaderView, QTextEdit, QProgressBar, QGroupBox,
    QDoubleSpinBox, QCheckBox, QComboBox, QDialog, QDialogButtonBox,
    QFrame, QAction, QAbstractItemView, QListWidget, QListWidgetItem,
    QSplitter, QLineEdit, QSizePolicy
)

APP_NAME = "Gene Expression Integration & Batch Correction Studio"
APP_VERSION = "1.1.0 - pyComBat Jupyter Edition"
DEVELOPERS = "Md Fazlay Rabbi and Enamul Haq Rana"
AFFILIATION = "Department of Zoology, University of Rajshahi, Bangladesh"
CONTACT_EMAIL = "rabbi.ms244@gmail.com"

# Publication-oriented typography.
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.labelsize": 13,
    "axes.labelweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10.5,
    "figure.titlesize": 17,
    "figure.titleweight": "bold",
    "axes.linewidth": 1.2,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.12,
})

USER_MANUAL_HTML = f"""
<h2>{APP_NAME}</h2>
<p><b>Version:</b> {APP_VERSION}</p>
<p><b>Developed by:</b> {DEVELOPERS}<br>
{AFFILIATION}<br>
<b>Contact:</b> {CONTACT_EMAIL}</p>
<hr>
<h3>Purpose</h3>
<p>This application integrates processed GEO microarray gene-expression datasets, provides dataset-specific
expression-scale review, applies user-approved log2 transformations, assesses dataset-associated batch structure,
and performs ComBat batch correction through pyComBat.</p>

<h3>Required input</h3>
<ul>
<li>Two or more processed microarray CSV files.</li>
<li>The filename must contain a GEO Series accession such as <code>GSE16879</code>.</li>
<li>Rows should represent genes and columns should represent samples.</li>
<li>The gene column should preferably be named <code>gene_symbol</code>; <code>ID</code> is also accepted.</li>
<li>Sample columns should follow the Part 1 naming convention: <code>control</code>, <code>control.1</code>, ... and
<code>disease</code>, <code>disease.1</code>, ... . The terms <code>healthy</code>, <code>normal</code>, <code>case</code>, and
<code>patient</code> are also recognized.</li>
</ul>

<h3>Recommended workflow</h3>
<ol>
<li><b>Data &amp; Log2 tab:</b> Click <b>Load GEO CSV Files</b>. Check the dataset table. The app reports GSE ID,
filename, genes, samples, Control/Disease counts, and expression-scale statistics.</li>
<li>Datasets flagged <b>Log2 recommended</b> are selected automatically. Review the selections and pseudocount.
Click <b>Apply Selected Log2 Transformations</b>. Transformations are always recalculated from the originally loaded
matrix, so clicking the button again does not double-transform the data.</li>
<li><b>Batch Correction tab:</b> Click <b>Assess Integration &amp; Batch Effect</b>. The application intersects common
genes, merges samples, calculates PCA-based QC inputs, dataset-origin silhouette score, and within-to-between batch
distance ratio, and shows a qualitative batch-correction recommendation.</li>
<li>The recommendation is guidance, not a formal statistical test. Review the PCA and both numerical metrics.
A positive dataset-origin silhouette indicates dataset-associated clustering. A within-to-between distance ratio far
below 1 indicates that samples from the same dataset are more similar to each other than to samples from other datasets.</li>
<li>If correction is appropriate, keep <b>Preserve biological group</b> checked and click <b>Run ComBat Correction</b>.
The app verifies that the Control/Disease covariate is not perfectly confounded with dataset before correction.</li>
<li><b>Figures tab:</b> Review PCA, expression boxplots, silhouette analysis, batch-distance analysis, and QC summary.
PCA legends and boxplot labels use only the GSE accession numbers.</li>
<li><b>Log &amp; Export tab:</b> Choose an output folder and click <b>Save All Results</b>.</li>
</ol>

<h3>Log2 guidance</h3>
<p>The scale detector is a conservative heuristic for processed microarray data. It provides a recommendation rather
than forcing a transformation. The user remains responsible for verifying dataset processing information in GEO.
If a selected dataset contains values for which <code>x + pseudocount &lt;= 0</code>, log2 transformation is stopped and
an explanatory error is shown.</p>

<h3>Batch-effect guidance</h3>
<p>The application labels batch correction as <b>recommended</b> when the descriptive dataset-origin QC metrics show
clear separation. The guidance uses the silhouette score and the within-to-between distance ratio and is intentionally
reported as a heuristic. There is no universal cutoff that proves whether batch correction is required. PCA and study
design should always be reviewed together with the numerical metrics.</p>

<h3>Outputs</h3>
<ul>
<li>Merged preprocessed common-gene expression matrix.</li>
<li>ComBat-corrected expression matrix when ComBat has been run.</li>
<li>Sample metadata and dataset summary tables.</li>
<li>Batch x biological-group table.</li>
<li>QC metrics and analysis settings.</li>
<li>Analysis log and a copy of this user manual.</li>
<li>Publication figures in PNG (600 DPI), TIFF (600 DPI), SVG, and PDF.</li>
</ul>

<h3>Important notes</h3>
<ul>
<li>ComBat should not be used blindly when biological condition is strongly or perfectly confounded with dataset.</li>
<li>Silhouette and distance-ratio values are descriptive batch-QC measures and should not be treated as independent
proof that all technical variation has been removed.</li>
<li>The standalone <code>combat</code> package is used here to reproduce the study workflow. The application includes
a NumPy 2.x compatibility shim for the legacy <code>np.mat</code> call used internally by pyComBat.</li>
</ul>
"""

# ============================================================
# 2. CORE UTILITIES
# ============================================================

def extract_gse_id(file_path):
    match = re.search(r"(GSE\d+)", Path(file_path).name, flags=re.IGNORECASE)
    if not match:
        raise ValueError(
            f"The filename '{Path(file_path).name}' does not contain a GEO Series accession. "
            "Rename the file so it contains a GSE number, e.g., GSE16879.csv."
        )
    return match.group(1).upper()


def infer_biological_group(column_name):
    text = str(column_name).strip().lower()
    base = text.split(".")[0].strip()
    if base in {"control", "healthy", "normal"}:
        return "control"
    if base in {"disease", "case", "patient"}:
        return "disease"
    if text.startswith(("control.", "healthy.", "normal.")):
        return "control"
    if text.startswith(("disease.", "case.", "patient.")):
        return "disease"
    return "unknown"


def determine_gene_column(df):
    for col in ["gene_symbol", "Gene_Symbol", "GENE_SYMBOL", "symbol", "Symbol", "ID"]:
        if col in df.columns:
            return col
    return df.columns[0]


def recommend_log2(expression_df):
    """Conservative processed-microarray scale recommendation; does not force transformation."""
    values = expression_df.to_numpy(dtype=float, copy=False).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return False, {}, "Manual review required"

    q01, q25, q50, q75, q99 = np.quantile(values, [0.01, 0.25, 0.50, 0.75, 0.99])
    vmin = float(np.min(values))
    vmax = float(np.max(values))
    spread = float(q99 - q01)

    recommended = bool((q99 > 100.0) or (vmax > 1000.0 and spread > 50.0))
    stats = {
        "min": vmin,
        "q01": float(q01),
        "q25": float(q25),
        "median": float(q50),
        "q75": float(q75),
        "q99": float(q99),
        "max": vmax,
    }
    label = "Log2 recommended" if recommended else "Log2 not indicated"
    return recommended, stats, label


def load_expression_csv(file_path):
    gse_id = extract_gse_id(file_path)
    raw = pd.read_csv(file_path)
    if raw.shape[1] < 2:
        raise ValueError(f"{gse_id}: at least one gene column and one sample column are required.")

    gene_col = determine_gene_column(raw)
    raw_gene = raw[gene_col]
    gene_series = raw_gene.astype(str).str.strip().str.upper()
    valid_gene = (
        raw_gene.notna()
        & (gene_series != "")
        & ~gene_series.isin({"NAN", "NONE", "NA", "N/A", "NULL"})
    )
    raw = raw.loc[valid_gene].copy()
    raw[gene_col] = gene_series.loc[valid_gene]

    sample_cols = [c for c in raw.columns if c != gene_col]
    numeric = raw[sample_cols].apply(pd.to_numeric, errors="coerce")
    total_cells = numeric.shape[0] * numeric.shape[1]
    n_missing = int(numeric.isna().sum().sum())
    if total_cells and n_missing / total_cells > 0.001:
        raise ValueError(
            f"{gse_id}: {n_missing:,} non-numeric/missing expression cells were detected. "
            "Verify that the file contains one gene column followed only by numeric sample columns."
        )

    finite_mask = np.isfinite(numeric.to_numpy(dtype=float)).all(axis=1)
    removed_nonfinite = int((~finite_mask).sum())
    raw = raw.loc[finite_mask].copy()
    numeric = numeric.loc[finite_mask].copy()

    matrix = numeric.copy()
    matrix.index = raw[gene_col].astype(str).values
    matrix.index.name = "gene_symbol"

    duplicate_rows = int(matrix.index.duplicated(keep=False).sum())
    if matrix.index.duplicated().any():
        matrix = matrix.groupby(level=0, sort=False).mean()

    groups = [infer_biological_group(c) for c in matrix.columns]
    unknown = [str(c) for c, g in zip(matrix.columns, groups) if g == "unknown"]
    if unknown:
        preview = ", ".join(unknown[:8])
        more = " ..." if len(unknown) > 8 else ""
        raise ValueError(
            f"{gse_id}: {len(unknown)} sample column(s) could not be classified as Control or Disease: "
            f"{preview}{more}.\nUse the Part 1 naming convention: control/control.1/... and disease/disease.1/... ."
        )

    n_control = int(sum(g == "control" for g in groups))
    n_disease = int(sum(g == "disease" for g in groups))
    if n_control == 0 or n_disease == 0:
        raise ValueError(
            f"{gse_id}: both biological groups are required. "
            f"Detected Control={n_control}, Disease={n_disease}."
        )

    log_recommended, scale_stats, recommendation = recommend_log2(matrix)

    return {
        "file_path": str(file_path),
        "file_name": Path(file_path).name,
        "gse": gse_id,
        "matrix": matrix,
        "working_matrix": None,
        "groups": groups,
        "genes": int(matrix.shape[0]),
        "samples": int(matrix.shape[1]),
        "control": n_control,
        "disease": n_disease,
        "duplicate_rows_collapsed": duplicate_rows,
        "nonfinite_rows_removed": removed_nonfinite,
        "log_recommended": log_recommended,
        "log2_applied": False,
        "scale_stats": scale_stats,
        "recommendation": recommendation,
        "transformation_status": "Pending",
    }


def apply_log2_transform(df, pseudocount):
    arr = df.to_numpy(dtype=float, copy=False)
    if np.nanmin(arr + pseudocount) <= 0:
        raise ValueError(
            "Log2 transformation cannot be applied because one or more values + pseudocount are <= 0. "
            "Change the pseudocount only if scientifically justified, or leave this dataset untransformed."
        )
    return np.log2(df + pseudocount)


def standardize_samples(expression_df):
    x = expression_df.T.to_numpy(dtype=np.float64, copy=True)
    return StandardScaler(copy=False).fit_transform(x)


def compute_distance_ratio(standardized_samples, batch_labels):
    labels = np.asarray(batch_labels)
    if standardized_samples.shape[0] < 2:
        return np.nan, np.nan, np.nan
    distances = pdist(standardized_samples, metric="euclidean")
    i, j = np.triu_indices(len(labels), k=1)
    same = labels[i] == labels[j]
    if not np.any(same) or np.all(same):
        return np.nan, np.nan, np.nan
    mean_within = float(np.mean(distances[same]))
    mean_between = float(np.mean(distances[~same]))
    ratio = float(mean_within / mean_between) if mean_between != 0 else np.nan
    return ratio, mean_within, mean_between


def compute_qc(expression_df, batch_labels):
    z = standardize_samples(expression_df)
    pca = PCA(n_components=2, random_state=0)
    coords = pca.fit_transform(z)
    variance = pca.explained_variance_ratio_ * 100.0

    unique_batches = np.unique(batch_labels)
    if len(unique_batches) >= 2 and len(batch_labels) > len(unique_batches):
        sil_mean = float(silhouette_score(z, batch_labels, metric="euclidean"))
        sil_samples = silhouette_samples(z, batch_labels, metric="euclidean")
    else:
        sil_mean = np.nan
        sil_samples = np.full(len(batch_labels), np.nan)

    ratio, mean_within, mean_between = compute_distance_ratio(z, batch_labels)
    return {
        "pca_coords": coords,
        "pca_variance": variance,
        "silhouette": sil_mean,
        "silhouette_samples": sil_samples,
        "distance_ratio": ratio,
        "mean_within": mean_within,
        "mean_between": mean_between,
    }


def design_is_confounded(batch_labels, biological_groups):
    batch_df = pd.get_dummies(pd.Series(batch_labels, dtype="category"), drop_first=False, dtype=float)
    group_df = pd.get_dummies(pd.Series(biological_groups, dtype="category"), drop_first=True, dtype=float)
    design = np.column_stack([batch_df.to_numpy(), group_df.to_numpy()])
    rank = np.linalg.matrix_rank(design)
    return rank < design.shape[1], rank, design.shape[1]


def deterministic_distribution_sample(df, max_values=100000):
    values = df.to_numpy(dtype=float, copy=False).ravel()
    values = values[np.isfinite(values)]
    if values.size <= max_values:
        return values
    idx = np.linspace(0, values.size - 1, max_values, dtype=int)
    return values[idx]


def batch_effect_guidance(silhouette_value, distance_ratio):
    """Qualitative, deliberately conservative guidance; not a formal significance test."""
    if not np.isfinite(silhouette_value) or not np.isfinite(distance_ratio):
        return "Manual review required", "review"
    if silhouette_value >= 0.20 or distance_ratio <= 0.80:
        return "ComBat recommended — strong dataset-associated structure detected", "strong"
    if silhouette_value >= 0.10 or distance_ratio <= 0.90:
        return "Consider ComBat — moderate dataset-associated structure detected", "moderate"
    return "No strong batch-associated separation detected — correction may be optional", "low"


def style_axis_bold(ax):
    ax.title.set_fontweight("bold")
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")
    leg = ax.get_legend()
    if leg is not None:
        for text in leg.get_texts():
            text.set_fontweight("bold")


def dataset_color_map(order):
    """Modern Matplotlib colormap API; avoids removed matplotlib.cm.get_cmap()."""
    n = max(20, len(order))
    cmap = matplotlib.colormaps["tab20"].resampled(n)
    return {gse: cmap(i) for i, gse in enumerate(order)}


def make_unique_sample_id(gse, group, group_counter):
    return f"{gse}__{group}_{group_counter:03d}"

# ============================================================
# 3. BACKGROUND WORKERS
# ============================================================
class DatasetLoaderWorker(QObject):
    progress = pyqtSignal(int, str)
    log = pyqtSignal(str)
    finished = pyqtSignal(object)
    failed = pyqtSignal(str)

    def __init__(self, paths):
        super().__init__()
        self.paths = list(paths)

    def run(self):
        try:
            datasets = []
            total = len(self.paths)
            for i, path in enumerate(self.paths, start=1):
                self.progress.emit(int((i - 1) / max(total, 1) * 90), f"Loading {Path(path).name}...")
                ds = load_expression_csv(path)
                datasets.append(ds)
                self.log.emit(
                    f"Loaded {ds['gse']}: {ds['genes']:,} unique genes × {ds['samples']:,} samples; "
                    f"Control={ds['control']}, Disease={ds['disease']}; {ds['recommendation']}."
                )
                self.progress.emit(int(i / max(total, 1) * 90), f"Loaded {ds['gse']}")

            gses = [d["gse"] for d in datasets]
            if len(set(gses)) != len(gses):
                raise ValueError(
                    "Duplicate GSE accessions were detected. Load only one processed file per GEO dataset."
                )
            self.progress.emit(100, "Dataset loading completed")
            self.finished.emit(datasets)
        except Exception:
            self.failed.emit(traceback.format_exc())


class PreprocessWorker(QObject):
    progress = pyqtSignal(int, str)
    log = pyqtSignal(str)
    finished = pyqtSignal(object)
    failed = pyqtSignal(str)

    def __init__(self, datasets, decisions, pseudocount):
        super().__init__()
        self.datasets = datasets
        self.decisions = decisions
        self.pseudocount = float(pseudocount)

    def run(self):
        try:
            total = len(self.datasets)
            for i, ds in enumerate(self.datasets, start=1):
                use_log = bool(self.decisions.get(ds["gse"], False))
                self.progress.emit(int((i - 1) / total * 90), f"Processing {ds['gse']}...")
                # Always start from the original loaded matrix: no cumulative/double log transformation.
                if use_log:
                    ds["working_matrix"] = apply_log2_transform(ds["matrix"], self.pseudocount)
                    ds["log2_applied"] = True
                    if ds["log_recommended"]:
                        ds["transformation_status"] = "Complete — log2 applied; no further log2 needed"
                    else:
                        ds["transformation_status"] = "Complete — log2 applied by user"
                    self.log.emit(f"{ds['gse']}: applied log2(x + {self.pseudocount:g}).")
                else:
                    ds["working_matrix"] = ds["matrix"].copy()
                    ds["log2_applied"] = False
                    if ds["log_recommended"]:
                        ds["transformation_status"] = "Original scale retained — recommendation overridden"
                        self.log.emit(f"{ds['gse']}: log2 was recommended, but original scale was retained by user choice.")
                    else:
                        ds["transformation_status"] = "Complete — no log2 needed"
                        self.log.emit(f"{ds['gse']}: no log2 transformation indicated; original scale retained.")
                self.progress.emit(int(i / total * 90), f"Processed {ds['gse']}")
            self.progress.emit(100, "Expression-scale processing completed")
            self.finished.emit(self.datasets)
        except Exception:
            self.failed.emit(traceback.format_exc())


class AssessmentWorker(QObject):
    progress = pyqtSignal(int, str)
    log = pyqtSignal(str)
    finished = pyqtSignal(object)
    failed = pyqtSignal(str)

    def __init__(self, datasets):
        super().__init__()
        self.datasets = datasets

    def run(self):
        try:
            if any(ds.get("working_matrix") is None for ds in self.datasets):
                raise RuntimeError("Run the log2/preprocessing step before assessing batch effects.")

            self.progress.emit(5, "Finding common genes across datasets...")
            common = set(self.datasets[0]["working_matrix"].index)
            for ds in self.datasets[1:]:
                common.intersection_update(ds["working_matrix"].index)
            if not common:
                raise ValueError("No common genes were found across the selected datasets.")
            common_genes = sorted(common)

            matrices = []
            metadata_rows = []
            batch_labels = []
            biological_groups = []
            dataset_order = []

            self.progress.emit(18, "Building integrated common-gene matrix...")
            for batch_index, ds in enumerate(self.datasets):
                gse = ds["gse"]
                dataset_order.append(gse)
                sub = ds["working_matrix"].loc[common_genes].copy()
                new_cols = []
                group_counters = {"control": 0, "disease": 0}
                for original_col, group in zip(sub.columns, ds["groups"]):
                    group_counters[group] += 1
                    sample_id = make_unique_sample_id(gse, group, group_counters[group])
                    new_cols.append(sample_id)
                    batch_labels.append(batch_index)
                    biological_groups.append(group)
                    metadata_rows.append({
                        "sample_id": sample_id,
                        "dataset": gse,
                        "original_column": str(original_col),
                        "biological_group": group,
                        "log2_applied": bool(ds["log2_applied"]),
                    })
                sub.columns = new_cols
                matrices.append(sub)

            merged = pd.concat(matrices, axis=1, join="inner")
            merged.index.name = "gene_symbol"

            finite_rows = np.isfinite(merged.to_numpy(dtype=float)).all(axis=1)
            removed_nonfinite = int((~finite_rows).sum())
            if removed_nonfinite:
                merged = merged.loc[finite_rows]
                self.log.emit(f"Removed {removed_nonfinite:,} common gene(s) containing non-finite values.")

            variances = merged.var(axis=1, ddof=0)
            variable_rows = variances > 1e-12
            removed_constant = int((~variable_rows).sum())
            if removed_constant:
                merged = merged.loc[variable_rows]
                self.log.emit(f"Removed {removed_constant:,} zero-variance gene(s).")
            if merged.empty:
                raise ValueError("No analyzable genes remained after integration quality checks.")

            metadata = pd.DataFrame(metadata_rows)
            batch_group_table = pd.crosstab(metadata["dataset"], metadata["biological_group"])
            self.log.emit(f"Integrated matrix: {merged.shape[0]:,} common genes × {merged.shape[1]:,} samples")
            self.log.emit("Batch × biological-group table:\n" + batch_group_table.to_string())

            self.progress.emit(45, "Calculating PCA, silhouette score and batch-distance ratio...")
            qc_before = compute_qc(merged, batch_labels)
            self.log.emit(
                f"Before correction: silhouette={qc_before['silhouette']:.4f}; "
                f"within/between distance ratio={qc_before['distance_ratio']:.4f}"
            )

            guidance_text, guidance_level = batch_effect_guidance(
                qc_before["silhouette"], qc_before["distance_ratio"]
            )
            self.log.emit("Batch-effect guidance: " + guidance_text)

            self.progress.emit(78, "Preparing expression-distribution summaries...")
            before_distributions = {}
            for gse in dataset_order:
                cols = metadata.loc[metadata["dataset"] == gse, "sample_id"].tolist()
                before_distributions[gse] = deterministic_distribution_sample(merged[cols])

            summary_rows = []
            for ds in self.datasets:
                summary_rows.append({
                    "dataset": ds["gse"],
                    "file_name": ds["file_name"],
                    "unique_genes_before_integration": ds["genes"],
                    "samples": ds["samples"],
                    "control_samples": ds["control"],
                    "disease_samples": ds["disease"],
                    "log2_recommended": ds["log_recommended"],
                    "log2_applied": ds["log2_applied"],
                    "transformation_status": ds["transformation_status"],
                    "input_min": ds["scale_stats"].get("min", np.nan),
                    "input_median": ds["scale_stats"].get("median", np.nan),
                    "input_q99": ds["scale_stats"].get("q99", np.nan),
                })

            result = {
                "merged": merged,
                "metadata": metadata,
                "batch_labels": np.asarray(batch_labels),
                "biological_groups": biological_groups,
                "dataset_order": dataset_order,
                "batch_group_table": batch_group_table,
                "qc_before": qc_before,
                "before_distributions": before_distributions,
                "dataset_summary": pd.DataFrame(summary_rows),
                "guidance_text": guidance_text,
                "guidance_level": guidance_level,
                "removed_nonfinite_common_genes": removed_nonfinite,
                "removed_zero_variance_genes": removed_constant,
            }
            self.progress.emit(100, "Batch-effect assessment completed")
            self.finished.emit(result)
        except Exception:
            self.failed.emit(traceback.format_exc())


class CombatWorker(QObject):
    progress = pyqtSignal(int, str)
    log = pyqtSignal(str)
    finished = pyqtSignal(object)
    failed = pyqtSignal(str)

    def __init__(self, assessment_result, preserve_group=True, par_prior=True, mean_only=False):
        super().__init__()
        self.result = assessment_result
        self.preserve_group = bool(preserve_group)
        self.par_prior = bool(par_prior)
        self.mean_only = bool(mean_only)

    def run(self):
        try:
            merged = self.result["merged"]
            batch_labels = self.result["batch_labels"].tolist()
            biological_groups = self.result["biological_groups"]

            if self.preserve_group:
                confounded, rank, columns = design_is_confounded(batch_labels, biological_groups)
                if confounded:
                    raise ValueError(
                        "Control/Disease is perfectly confounded with dataset/batch "
                        f"(design rank {rank} < {columns}). ComBat cannot reliably separate these effects. "
                        "Review the study design before continuing."
                    )
                self.log.emit("ComBat design check passed: biological group is not perfectly confounded with batch.")

            self.progress.emit(12, "Running pyComBat batch correction...")
            mod = biological_groups if self.preserve_group else []
            capture = io.StringIO()
            with redirect_stdout(capture), redirect_stderr(capture):
                corrected = pycombat(
                    merged,
                    batch_labels,
                    mod=mod,
                    par_prior=self.par_prior,
                    mean_only=self.mean_only,
                )
            combat_output = capture.getvalue().strip()
            if combat_output:
                for line in combat_output.splitlines():
                    if line.strip():
                        self.log.emit("pyComBat: " + line.strip())

            if not isinstance(corrected, pd.DataFrame):
                corrected = pd.DataFrame(corrected, index=merged.index, columns=merged.columns)
            corrected.index.name = "gene_symbol"
            if corrected.shape != merged.shape:
                raise RuntimeError(f"Unexpected pyComBat output shape {corrected.shape}; expected {merged.shape}.")
            if not np.isfinite(corrected.to_numpy(dtype=float)).all():
                raise RuntimeError("pyComBat returned non-finite values; correction was stopped.")

            self.progress.emit(63, "Calculating post-correction QC metrics...")
            qc_after = compute_qc(corrected, batch_labels)
            self.log.emit(
                f"After correction: silhouette={qc_after['silhouette']:.4f}; "
                f"within/between distance ratio={qc_after['distance_ratio']:.4f}"
            )

            self.progress.emit(82, "Preparing post-correction expression distributions...")
            metadata = self.result["metadata"]
            after_distributions = {}
            for gse in self.result["dataset_order"]:
                cols = metadata.loc[metadata["dataset"] == gse, "sample_id"].tolist()
                after_distributions[gse] = deterministic_distribution_sample(corrected[cols])

            metrics = {
                "silhouette_before": self.result["qc_before"]["silhouette"],
                "silhouette_after": qc_after["silhouette"],
                "distance_ratio_before": self.result["qc_before"]["distance_ratio"],
                "distance_ratio_after": qc_after["distance_ratio"],
                "mean_within_before": self.result["qc_before"]["mean_within"],
                "mean_between_before": self.result["qc_before"]["mean_between"],
                "mean_within_after": qc_after["mean_within"],
                "mean_between_after": qc_after["mean_between"],
            }

            sil_improved = abs(metrics["silhouette_after"]) < abs(metrics["silhouette_before"])
            ratio_improved = abs(metrics["distance_ratio_after"] - 1.0) < abs(metrics["distance_ratio_before"] - 1.0)
            if sil_improved and ratio_improved:
                post_guidance = "Post-correction QC improved in both monitored batch-structure metrics"
                post_level = "good"
            elif sil_improved or ratio_improved:
                post_guidance = "Post-correction QC improved in one metric; review PCA and all QC outputs"
                post_level = "review"
            else:
                post_guidance = "Post-correction QC did not improve in the expected direction; review settings and study design"
                post_level = "warning"
            self.log.emit(post_guidance + ".")

            self.result = dict(self.result)
            self.result.update({
                "corrected": corrected,
                "qc_after": qc_after,
                "after_distributions": after_distributions,
                "metrics": metrics,
                "post_guidance": post_guidance,
                "post_level": post_level,
                "analysis_settings": {
                    "app_name": APP_NAME,
                    "app_version": APP_VERSION,
                    "created_at": datetime.now().isoformat(timespec="seconds"),
                    "batch_method": "ComBat implemented through pyComBat (combat package)",
                    "preserve_biological_group": self.preserve_group,
                    "parametric_prior": self.par_prior,
                    "mean_only": self.mean_only,
                    "common_genes_retained": int(merged.shape[0]),
                    "total_samples": int(merged.shape[1]),
                    "datasets": self.result["dataset_order"],
                    "log2_decisions": self.result["dataset_summary"][["dataset", "log2_recommended", "log2_applied"]].to_dict(orient="records"),
                },
            })
            self.progress.emit(100, "ComBat correction completed")
            self.finished.emit(self.result)
        except Exception:
            self.failed.emit(traceback.format_exc())

# ============================================================
# 4. DIALOG / PLOT PANEL
# ============================================================
class TextDialog(QDialog):
    def __init__(self, title, html, parent=None, width=900, height=700):
        super().__init__(parent)
        self.setWindowTitle(title)
        self.resize(width, height)
        layout = QVBoxLayout(self)
        browser = QTextEdit()
        browser.setReadOnly(True)
        browser.setHtml(html)
        browser.setStyleSheet("font-size: 11pt; padding: 10px;")
        layout.addWidget(browser)
        buttons = QDialogButtonBox(QDialogButtonBox.Close)
        buttons.button(QDialogButtonBox.Close).clicked.connect(self.close)
        layout.addWidget(buttons)


class PlotPanel(QWidget):
    """Compact figure preview; the main setup pages remain uncluttered."""
    def __init__(self, parent=None):
        super().__init__(parent)
        self.current_canvas = None
        self.current_toolbar = None
        self.layout = QVBoxLayout(self)
        self.layout.setContentsMargins(6, 6, 6, 6)
        self.placeholder = QLabel("Run batch-effect assessment / ComBat, then select a figure from the list.")
        self.placeholder.setAlignment(Qt.AlignCenter)
        self.placeholder.setStyleSheet("color:#66788c;font-weight:600;font-size:11pt;")
        self.layout.addWidget(self.placeholder, 1)

    def show_figure(self, figure):
        if self.placeholder is not None:
            self.placeholder.deleteLater()
            self.placeholder = None
        if self.current_toolbar is not None:
            self.layout.removeWidget(self.current_toolbar)
            self.current_toolbar.deleteLater()
            self.current_toolbar = None
        if self.current_canvas is not None:
            self.layout.removeWidget(self.current_canvas)
            self.current_canvas.deleteLater()
            self.current_canvas = None

        self.current_canvas = FigureCanvas(figure)
        self.current_canvas.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.current_canvas.setMinimumSize(700, 430)
        self.current_toolbar = NavigationToolbar(self.current_canvas, self)
        self.layout.addWidget(self.current_toolbar)
        self.layout.addWidget(self.current_canvas, 1)
        self.current_canvas.draw_idle()

# ============================================================
# 5. MAIN APPLICATION
# ============================================================
class GeneExpressionBatchStudio(QMainWindow):
    def __init__(self):
        super().__init__()
        self.datasets = []
        self.preprocessing_ready = False
        self.assessment_result = None
        self.analysis_result = None
        self.figure_store = {}
        self.output_directory = None
        self._updating_table = False
        self._threads = []

        self.setWindowTitle(APP_NAME)
        self.resize(1480, 900)
        self.setMinimumSize(1180, 760)

        self._apply_theme()
        self._build_menu()
        self._build_ui()
        self.log_message("Application ready. Load two or more processed GEO microarray CSV files to begin.")

    # -------------------------- UI --------------------------
    def _apply_theme(self):
        self.setStyleSheet("""
            QMainWindow{background:#eef3f8;}
            QWidget{font-family:'Segoe UI';font-size:10.2pt;color:#182230;}
            QGroupBox{font-weight:700;border:1px solid #d7e0ea;border-radius:10px;margin-top:10px;padding:13px 10px 10px 10px;background:white;}
            QGroupBox::title{subcontrol-origin:margin;left:14px;padding:0 6px;color:#17324d;}
            QPushButton{background:#245b9e;color:white;border:none;border-radius:7px;padding:9px 14px;font-weight:700;}
            QPushButton:hover{background:#1b4a83;}
            QPushButton#run{background:#14804a;} QPushButton#run:hover{background:#0f683c;}
            QPushButton#export{background:#6d43b5;} QPushButton#export:hover{background:#583595;}
            QPushButton#secondary{background:#475569;} QPushButton#secondary:hover{background:#334155;}
            QPushButton#warning{background:#d97706;} QPushButton#warning:hover{background:#b45f05;}
            QPushButton:disabled{background:#aeb8c5;color:#eef2f6;}
            QLineEdit,QDoubleSpinBox,QComboBox{background:white;border:1px solid #cbd6e2;border-radius:6px;padding:7px;min-height:20px;}
            QLineEdit:focus,QDoubleSpinBox:focus,QComboBox:focus{border:1px solid #2e73b8;}
            QTabWidget::pane{border:1px solid #d7e0ea;border-radius:8px;background:white;}
            QTabBar::tab{padding:10px 16px;background:#e7edf4;margin-right:2px;border-top-left-radius:5px;border-top-right-radius:5px;}
            QTabBar::tab:selected{background:white;font-weight:700;color:#17324d;}
            QTableWidget{gridline-color:#e5ebf1;background:white;selection-background-color:#dcecff;alternate-background-color:#f8fafc;}
            QHeaderView::section{background:#eef3f8;padding:7px;border:0;border-right:1px solid #d7e0ea;font-weight:700;}
            QTextEdit{background:white;border:1px solid #d7e0ea;border-radius:7px;}
            QListWidget{background:white;border:1px solid #d7e0ea;border-radius:7px;padding:4px;}
            QListWidget::item{padding:9px;border-radius:5px;} QListWidget::item:selected{background:#dcecff;color:#17324d;font-weight:700;}
            QProgressBar{border:1px solid #cbd6e2;border-radius:6px;text-align:center;background:white;min-height:21px;font-weight:700;}
            QProgressBar::chunk{background:#2e73b8;border-radius:5px;}
            QCheckBox{spacing:7px;}
        """)

    def _build_menu(self):
        help_menu = self.menuBar().addMenu("Help")
        manual_action = QAction("User Manual", self)
        manual_action.triggered.connect(self.show_user_manual)
        help_menu.addAction(manual_action)
        about_action = QAction("About", self)
        about_action.triggered.connect(self.show_about)
        help_menu.addAction(about_action)

    def _build_ui(self):
        root_widget = QWidget()
        self.setCentralWidget(root_widget)
        main = QVBoxLayout(root_widget)
        main.setContentsMargins(16, 14, 16, 12)
        main.setSpacing(10)

        header = QFrame()
        header.setStyleSheet(
            "QFrame{background:qlineargradient(x1:0,y1:0,x2:1,y2:0,stop:0 #0f2742,stop:0.58 #173b61,stop:1 #245b9e);border-radius:12px;}"
        )
        hl = QVBoxLayout(header)
        hl.setContentsMargins(22, 15, 22, 15)
        hl.setSpacing(4)
        title = QLabel(APP_NAME)
        title.setAlignment(Qt.AlignCenter)
        title.setStyleSheet("color:white;font-size:24px;font-weight:800;letter-spacing:0.2px;")
        sub = QLabel("Dataset-specific scale review • common-gene integration • pyComBat correction • batch-QC assessment")
        sub.setAlignment(Qt.AlignCenter)
        sub.setStyleSheet("color:#dbe8f6;font-size:11.2pt;font-weight:600;")
        dev = QLabel(f"Developed by {DEVELOPERS}  |  {AFFILIATION}")
        dev.setAlignment(Qt.AlignCenter)
        dev.setStyleSheet("color:#bcd0e6;font-size:10.2pt;font-weight:600;")
        hl.addWidget(title)
        hl.addWidget(sub)
        hl.addWidget(dev)
        main.addWidget(header)

        self.tabs = QTabWidget()
        main.addWidget(self.tabs, 1)
        self.data_tab = QWidget()
        self.batch_tab = QWidget()
        self.figures_tab = QWidget()
        self.log_export_tab = QWidget()
        self.tabs.addTab(self.data_tab, "1  Data & Log2")
        self.tabs.addTab(self.batch_tab, "2  Batch Correction")
        self.tabs.addTab(self.figures_tab, "3  Figures")
        self.tabs.addTab(self.log_export_tab, "4  Log & Export")

        self._build_data_tab()
        self._build_batch_tab()
        self._build_figures_tab()
        self._build_log_export_tab()

        action = QFrame()
        action.setStyleSheet("QFrame{background:#ffffff;border:1px solid #d7e0ea;border-radius:9px;}")
        al = QHBoxLayout(action)
        al.setContentsMargins(12, 8, 12, 8)
        self.progress = QProgressBar()
        self.progress.setRange(0, 100)
        self.progress.setValue(0)
        self.progress.setFormat("%p%")
        self.status_label = QLabel("Ready")
        self.status_label.setStyleSheet("font-weight:700;color:#476176;")
        manual_btn = QPushButton("User Manual")
        manual_btn.setObjectName("secondary")
        manual_btn.clicked.connect(self.show_user_manual)
        about_btn = QPushButton("About")
        about_btn.setObjectName("secondary")
        about_btn.clicked.connect(self.show_about)
        al.addWidget(self.progress, 1)
        al.addWidget(self.status_label)
        al.addStretch()
        al.addWidget(manual_btn)
        al.addWidget(about_btn)
        main.addWidget(action)

    def _build_data_tab(self):
        lay = QVBoxLayout(self.data_tab)
        lay.setContentsMargins(14, 14, 14, 14)
        lay.setSpacing(10)

        inp = QGroupBox("Processed GEO microarray datasets")
        il = QHBoxLayout(inp)
        self.load_btn = QPushButton("Load GEO CSV Files")
        self.load_btn.clicked.connect(self.select_files)
        self.clear_btn = QPushButton("Clear / Start New")
        self.clear_btn.setObjectName("secondary")
        self.clear_btn.clicked.connect(self.clear_all)
        self.dataset_summary_label = QLabel("No datasets loaded")
        self.dataset_summary_label.setStyleSheet("font-weight:700;color:#476176;")
        il.addWidget(self.load_btn)
        il.addWidget(self.clear_btn)
        il.addSpacing(10)
        il.addWidget(self.dataset_summary_label)
        il.addStretch()
        lay.addWidget(inp)

        table_box = QGroupBox("Dataset review and expression-scale recommendation")
        tl = QVBoxLayout(table_box)
        note = QLabel(
            "Each file is shown separately below. Datasets flagged as Log2 recommended are selected automatically. "
            "Plots later use only the GSE accession as the dataset label."
        )
        note.setWordWrap(True)
        note.setStyleSheet("color:#53677c;")
        tl.addWidget(note)

        self.dataset_table = QTableWidget(0, 12)
        self.dataset_table.setHorizontalHeaderLabels([
            "GSE", "Input file", "Genes", "Samples", "Control", "Disease",
            "Min", "Median", "99th %", "Scale assessment", "Apply log2", "Transformation status"
        ])
        self.dataset_table.setAlternatingRowColors(True)
        self.dataset_table.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.dataset_table.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.dataset_table.verticalHeader().setVisible(False)
        self.dataset_table.setMinimumHeight(315)
        hdr = self.dataset_table.horizontalHeader()
        for col in [0, 2, 3, 4, 5, 6, 7, 8, 10]:
            hdr.setSectionResizeMode(col, QHeaderView.ResizeToContents)
        hdr.setSectionResizeMode(1, QHeaderView.Stretch)
        hdr.setSectionResizeMode(9, QHeaderView.Stretch)
        hdr.setSectionResizeMode(11, QHeaderView.Stretch)
        self.dataset_table.itemChanged.connect(self._dataset_table_changed)
        tl.addWidget(self.dataset_table)
        lay.addWidget(table_box, 1)

        prep = QGroupBox("Apply selected log2 transformations")
        pl = QHBoxLayout(prep)
        pl.addWidget(QLabel("Pseudocount"))
        self.pseudocount = QDoubleSpinBox()
        self.pseudocount.setDecimals(4)
        self.pseudocount.setRange(0.0001, 100000.0)
        self.pseudocount.setValue(1.0)
        self.pseudocount.setSingleStep(0.5)
        pl.addWidget(self.pseudocount)
        self.restore_btn = QPushButton("Restore Automatic Selections")
        self.restore_btn.setObjectName("secondary")
        self.restore_btn.clicked.connect(self.restore_recommendations)
        self.apply_log_btn = QPushButton("Apply Selected Log2 Transformations")
        self.apply_log_btn.setObjectName("run")
        self.apply_log_btn.setEnabled(False)
        self.apply_log_btn.clicked.connect(self.start_preprocessing)
        self.preprocessing_badge = QLabel("Status: waiting for datasets")
        self.preprocessing_badge.setStyleSheet("font-weight:700;color:#6b7280;")
        pl.addWidget(self.restore_btn)
        pl.addWidget(self.apply_log_btn)
        pl.addSpacing(10)
        pl.addWidget(self.preprocessing_badge)
        pl.addStretch()
        lay.addWidget(prep)

    def _build_batch_tab(self):
        lay = QVBoxLayout(self.batch_tab)
        lay.setContentsMargins(14, 14, 14, 14)
        lay.setSpacing(10)

        assess = QGroupBox("Integration and pre-correction batch-effect assessment")
        ag = QGridLayout(assess)
        self.assess_btn = QPushButton("Assess Integration & Batch Effect")
        self.assess_btn.setObjectName("run")
        self.assess_btn.setEnabled(False)
        self.assess_btn.clicked.connect(self.start_assessment)
        ag.addWidget(self.assess_btn, 0, 0, 1, 2)

        self.common_genes_value = self._metric_card("Common genes", "—")
        self.samples_value = self._metric_card("Total samples", "—")
        self.silhouette_value = self._metric_card("Batch silhouette", "—")
        self.ratio_value = self._metric_card("Within / between ratio", "—")
        ag.addWidget(self.common_genes_value, 1, 0)
        ag.addWidget(self.samples_value, 1, 1)
        ag.addWidget(self.silhouette_value, 1, 2)
        ag.addWidget(self.ratio_value, 1, 3)

        self.batch_guidance = QLabel("Complete the Data & Log2 step, then assess batch-associated structure.")
        self.batch_guidance.setWordWrap(True)
        self.batch_guidance.setStyleSheet("padding:10px;border-radius:7px;background:#eef3f8;font-weight:700;color:#476176;")
        ag.addWidget(self.batch_guidance, 2, 0, 1, 4)
        lay.addWidget(assess)

        design_box = QGroupBox("Dataset × biological-group design")
        dl = QVBoxLayout(design_box)
        self.design_table = QTableWidget(0, 3)
        self.design_table.setHorizontalHeaderLabels(["GSE", "Control", "Disease"])
        self.design_table.verticalHeader().setVisible(False)
        self.design_table.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.design_table.setMaximumHeight(225)
        dh = self.design_table.horizontalHeader()
        dh.setSectionResizeMode(0, QHeaderView.Stretch)
        dh.setSectionResizeMode(1, QHeaderView.ResizeToContents)
        dh.setSectionResizeMode(2, QHeaderView.ResizeToContents)
        dl.addWidget(self.design_table)
        lay.addWidget(design_box)

        combat_box = QGroupBox("ComBat batch correction")
        cl = QGridLayout(combat_box)
        self.method_combo = QComboBox()
        self.method_combo.addItems(["ComBat (pyComBat) — study-compatible", "No correction — retain merged matrix"])
        self.method_combo.currentIndexChanged.connect(self._method_changed)
        cl.addWidget(QLabel("Correction method"), 0, 0)
        cl.addWidget(self.method_combo, 0, 1, 1, 3)

        self.preserve_group = QCheckBox("Preserve biological group (Control/Disease) during ComBat")
        self.preserve_group.setChecked(True)
        self.par_prior = QCheckBox("Parametric empirical-Bayes prior")
        self.par_prior.setChecked(True)
        self.mean_only = QCheckBox("Mean-only correction")
        self.mean_only.setChecked(False)
        cl.addWidget(self.preserve_group, 1, 0, 1, 2)
        cl.addWidget(self.par_prior, 1, 2)
        cl.addWidget(self.mean_only, 1, 3)

        self.combat_btn = QPushButton("Run ComBat Correction")
        self.combat_btn.setObjectName("run")
        self.combat_btn.setEnabled(False)
        self.combat_btn.clicked.connect(self.start_combat)
        cl.addWidget(self.combat_btn, 2, 0, 1, 2)
        self.post_qc_label = QLabel("Post-correction QC: not run")
        self.post_qc_label.setWordWrap(True)
        self.post_qc_label.setStyleSheet("font-weight:700;color:#6b7280;")
        cl.addWidget(self.post_qc_label, 2, 2, 1, 2)
        lay.addWidget(combat_box)
        lay.addStretch()

    def _build_figures_tab(self):
        lay = QVBoxLayout(self.figures_tab)
        lay.setContentsMargins(14, 14, 14, 14)
        lay.setSpacing(8)
        info = QLabel(
            "Select a figure to preview. The preview is intentionally compact; exported PNG/TIFF files are 600 DPI and SVG/PDF remain publication-ready."
        )
        info.setWordWrap(True)
        info.setStyleSheet("color:#53677c;")
        lay.addWidget(info)

        split = QSplitter(Qt.Horizontal)
        self.figure_list = QListWidget()
        self.figure_list.setMaximumWidth(270)
        self.figure_list.setMinimumWidth(230)
        self.figure_list.currentItemChanged.connect(self._figure_selected)
        self.plot_panel = PlotPanel()
        split.addWidget(self.figure_list)
        split.addWidget(self.plot_panel)
        split.setStretchFactor(0, 0)
        split.setStretchFactor(1, 1)
        split.setSizes([250, 1000])
        lay.addWidget(split, 1)

    def _build_log_export_tab(self):
        lay = QVBoxLayout(self.log_export_tab)
        lay.setContentsMargins(14, 14, 14, 14)
        lay.setSpacing(10)

        export_box = QGroupBox("Export all results")
        el = QGridLayout(export_box)
        self.output_path = QLineEdit()
        self.output_path.setReadOnly(True)
        self.output_path.setPlaceholderText("Choose a folder for matrices, QC tables, figures, reports, and user manual")
        choose_btn = QPushButton("Choose Location")
        choose_btn.setObjectName("secondary")
        choose_btn.clicked.connect(self.choose_output_directory)
        self.save_btn = QPushButton("Save All Results")
        self.save_btn.setObjectName("export")
        self.save_btn.setEnabled(False)
        self.save_btn.clicked.connect(self.save_all_results)
        el.addWidget(QLabel("Output folder"), 0, 0)
        el.addWidget(self.output_path, 0, 1)
        el.addWidget(choose_btn, 0, 2)
        el.addWidget(self.save_btn, 0, 3)
        fmt = QLabel("Figures: PNG 600 DPI • TIFF 600 DPI • SVG • PDF")
        fmt.setStyleSheet("color:#53677c;font-weight:600;")
        el.addWidget(fmt, 1, 1, 1, 3)
        el.setColumnStretch(1, 1)
        lay.addWidget(export_box)

        log_box = QGroupBox("Live analysis log")
        ll = QVBoxLayout(log_box)
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 9))
        self.log_text.setMinimumHeight(330)
        ll.addWidget(self.log_text)
        lay.addWidget(log_box, 1)

    def _metric_card(self, title, value):
        frame = QFrame()
        frame.setStyleSheet("QFrame{background:#f8fbff;border:1px solid #d7e0ea;border-radius:8px;padding:8px;}")
        layout = QVBoxLayout(frame)
        t = QLabel(title)
        t.setAlignment(Qt.AlignCenter)
        t.setStyleSheet("color:#60758a;font-weight:700;")
        v = QLabel(value)
        v.setObjectName("metric_value")
        v.setAlignment(Qt.AlignCenter)
        v.setStyleSheet("color:#17324d;font-size:17px;font-weight:800;")
        layout.addWidget(t)
        layout.addWidget(v)
        frame.value_label = v
        return frame

    # -------------------------- State / logging --------------------------
    def log_message(self, message):
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}] {message}")
        cursor = self.log_text.textCursor()
        cursor.movePosition(cursor.End)
        self.log_text.setTextCursor(cursor)

    def set_progress(self, value, message):
        self.progress.setValue(max(0, min(100, int(value))))
        self.status_label.setText(str(message))

    def _track_thread(self, thread):
        self._threads.append(thread)
        thread.finished.connect(lambda: self._threads.remove(thread) if thread in self._threads else None)

    def _set_busy(self, busy):
        self.load_btn.setEnabled(not busy)
        self.clear_btn.setEnabled(not busy)
        self.apply_log_btn.setEnabled((not busy) and bool(self.datasets))
        self.assess_btn.setEnabled((not busy) and self.preprocessing_ready)
        self.combat_btn.setEnabled((not busy) and self.assessment_result is not None and self.method_combo.currentIndex() == 0)
        self.save_btn.setEnabled((not busy) and self.assessment_result is not None)

    def clear_all(self):
        if self._threads:
            QMessageBox.information(self, "Busy", "Wait for the current operation to finish before starting a new analysis.")
            return
        self.datasets = []
        self.preprocessing_ready = False
        self.assessment_result = None
        self.analysis_result = None
        self.figure_store = {}
        self.output_directory = None
        self.output_path.clear()
        self._updating_table = True
        self.dataset_table.setRowCount(0)
        self._updating_table = False
        self.design_table.setRowCount(0)
        self.figure_list.clear()
        self.dataset_summary_label.setText("No datasets loaded")
        self.preprocessing_badge.setText("Status: waiting for datasets")
        self.preprocessing_badge.setStyleSheet("font-weight:700;color:#6b7280;")
        self.batch_guidance.setText("Complete the Data & Log2 step, then assess batch-associated structure.")
        self.batch_guidance.setStyleSheet("padding:10px;border-radius:7px;background:#eef3f8;font-weight:700;color:#476176;")
        for card in [self.common_genes_value, self.samples_value, self.silhouette_value, self.ratio_value]:
            card.value_label.setText("—")
        self.post_qc_label.setText("Post-correction QC: not run")
        self.apply_log_btn.setEnabled(False)
        self.assess_btn.setEnabled(False)
        self.combat_btn.setEnabled(False)
        self.save_btn.setEnabled(False)
        self.progress.setValue(0)
        self.status_label.setText("Ready")
        self.log_text.clear()
        self.log_message("New analysis started.")

    # -------------------------- Dataset loading --------------------------
    def select_files(self):
        paths, _ = QFileDialog.getOpenFileNames(
            self,
            "Select processed GEO microarray expression CSV files",
            "",
            "CSV files (*.csv);;All files (*.*)",
        )
        if not paths:
            return
        if len(paths) < 2:
            QMessageBox.warning(self, "Need multiple datasets", "Please select at least two GEO datasets.")
            return

        self.datasets = []
        self.preprocessing_ready = False
        self.assessment_result = None
        self.analysis_result = None
        self.figure_store = {}
        self.figure_list.clear()
        self.design_table.setRowCount(0)
        self._updating_table = True
        self.dataset_table.setRowCount(0)
        self._updating_table = False
        self.log_message(f"Loading {len(paths)} dataset(s)...")
        self.set_progress(0, "Loading datasets...")
        self._set_busy(True)

        thread = QThread()
        worker = DatasetLoaderWorker(paths)
        thread._worker = worker  # keep Python reference alive until the QThread finishes
        worker.moveToThread(thread)
        thread.started.connect(worker.run)
        worker.progress.connect(self.set_progress)
        worker.log.connect(self.log_message)
        worker.finished.connect(self._datasets_loaded)
        worker.failed.connect(self._worker_failed)
        worker.finished.connect(thread.quit)
        worker.failed.connect(thread.quit)
        thread.finished.connect(worker.deleteLater)
        thread.finished.connect(thread.deleteLater)
        self._track_thread(thread)
        thread.start()

    def _datasets_loaded(self, datasets):
        self.datasets = datasets
        self._updating_table = True
        self.dataset_table.setRowCount(len(datasets))
        total_samples = 0
        total_controls = 0
        total_disease = 0

        for row, ds in enumerate(datasets):
            total_samples += ds["samples"]
            total_controls += ds["control"]
            total_disease += ds["disease"]
            values = [
                ds["gse"], ds["file_name"], f"{ds['genes']:,}", f"{ds['samples']:,}",
                str(ds["control"]), str(ds["disease"]),
                f"{ds['scale_stats']['min']:.3g}", f"{ds['scale_stats']['median']:.3g}",
                f"{ds['scale_stats']['q99']:.3g}", ds["recommendation"],
            ]
            for col, val in enumerate(values):
                item = QTableWidgetItem(val)
                item.setTextAlignment(Qt.AlignCenter if col != 1 else Qt.AlignLeft | Qt.AlignVCenter)
                if col == 9:
                    if ds["log_recommended"]:
                        item.setBackground(QColor("#fff2cc"))
                        item.setForeground(QColor("#8a5a00"))
                    else:
                        item.setBackground(QColor("#e5f5e9"))
                        item.setForeground(QColor("#176b35"))
                self.dataset_table.setItem(row, col, item)

            check_item = QTableWidgetItem()
            check_item.setFlags(Qt.ItemIsEnabled | Qt.ItemIsUserCheckable)
            check_item.setCheckState(Qt.Checked if ds["log_recommended"] else Qt.Unchecked)
            check_item.setTextAlignment(Qt.AlignCenter)
            self.dataset_table.setItem(row, 10, check_item)

            status = QTableWidgetItem("Pending — click Apply")
            status.setTextAlignment(Qt.AlignCenter)
            status.setBackground(QColor("#eef3f8"))
            self.dataset_table.setItem(row, 11, status)

        self._updating_table = False
        self.dataset_summary_label.setText(
            f"{len(datasets)} datasets | {total_samples:,} samples | Control={total_controls:,} | Disease={total_disease:,}"
        )
        self.preprocessing_badge.setText("Status: recommendations ready — review and click Apply")
        self.preprocessing_badge.setStyleSheet("font-weight:700;color:#9a6700;")
        self.apply_log_btn.setEnabled(True)
        self.assess_btn.setEnabled(False)
        self.combat_btn.setEnabled(False)
        self.set_progress(100, "Datasets loaded — review log2 selections")
        self._set_busy(False)

    def _dataset_table_changed(self, item):
        if self._updating_table or item.column() != 10 or not self.datasets:
            return
        # A changed checkbox invalidates any previous transformation/assessment.
        self.preprocessing_ready = False
        self.assessment_result = None
        self.analysis_result = None
        self.figure_store = {}
        self.figure_list.clear()
        self.assess_btn.setEnabled(False)
        self.combat_btn.setEnabled(False)
        self.save_btn.setEnabled(False)
        self.preprocessing_badge.setText("Status: selection changed — click Apply again")
        self.preprocessing_badge.setStyleSheet("font-weight:700;color:#9a6700;")
        status = self.dataset_table.item(item.row(), 11)
        if status:
            status.setText("Pending — selection changed")
            status.setBackground(QColor("#fff2cc"))

    def restore_recommendations(self):
        if not self.datasets:
            return
        self._updating_table = True
        for row, ds in enumerate(self.datasets):
            item = self.dataset_table.item(row, 10)
            item.setCheckState(Qt.Checked if ds["log_recommended"] else Qt.Unchecked)
            status = self.dataset_table.item(row, 11)
            status.setText("Pending — click Apply")
            status.setBackground(QColor("#eef3f8"))
        self._updating_table = False
        self.preprocessing_ready = False
        self.assessment_result = None
        self.analysis_result = None
        self.figure_store = {}
        self.figure_list.clear()
        self.assess_btn.setEnabled(False)
        self.combat_btn.setEnabled(False)
        self.save_btn.setEnabled(False)
        self.preprocessing_badge.setText("Status: automatic selections restored — click Apply")
        self.preprocessing_badge.setStyleSheet("font-weight:700;color:#9a6700;")
        self.log_message("Log2 selections restored to the automatic recommendations.")

    def transform_decisions(self):
        decisions = {}
        for row, ds in enumerate(self.datasets):
            item = self.dataset_table.item(row, 10)
            decisions[ds["gse"]] = bool(item and item.checkState() == Qt.Checked)
        return decisions

    # -------------------------- Log2 preprocessing --------------------------
    def start_preprocessing(self):
        if not self.datasets:
            return
        decisions = self.transform_decisions()
        declined = [d["gse"] for d in self.datasets if d["log_recommended"] and not decisions[d["gse"]]]
        if declined:
            answer = QMessageBox.question(
                self,
                "Log2 recommendation overridden",
                "Log2 transformation was recommended for:\n\n"
                + "\n".join(declined)
                + "\n\nYou chose to retain the original scale for these dataset(s). Continue?",
                QMessageBox.Yes | QMessageBox.No,
                QMessageBox.No,
            )
            if answer != QMessageBox.Yes:
                return

        self.preprocessing_ready = False
        self.assessment_result = None
        self.analysis_result = None
        self.figure_store = {}
        self.figure_list.clear()
        self.log_message("Applying expression-scale decisions...")
        self.set_progress(0, "Applying log2 decisions...")
        self._set_busy(True)

        thread = QThread()
        worker = PreprocessWorker(self.datasets, decisions, self.pseudocount.value())
        thread._worker = worker  # keep Python reference alive until the QThread finishes
        worker.moveToThread(thread)
        thread.started.connect(worker.run)
        worker.progress.connect(self.set_progress)
        worker.log.connect(self.log_message)
        worker.finished.connect(self._preprocessing_finished)
        worker.failed.connect(self._worker_failed)
        worker.finished.connect(thread.quit)
        worker.failed.connect(thread.quit)
        thread.finished.connect(worker.deleteLater)
        thread.finished.connect(thread.deleteLater)
        self._track_thread(thread)
        thread.start()

    def _preprocessing_finished(self, datasets):
        self.datasets = datasets
        self.preprocessing_ready = True
        self._updating_table = True
        for row, ds in enumerate(datasets):
            status = self.dataset_table.item(row, 11)
            status.setText(ds["transformation_status"])
            if ds["log2_applied"] or not ds["log_recommended"]:
                status.setBackground(QColor("#e5f5e9"))
                status.setForeground(QColor("#176b35"))
            else:
                status.setBackground(QColor("#fff2cc"))
                status.setForeground(QColor("#8a5a00"))
        self._updating_table = False
        self.preprocessing_badge.setText("Status: preprocessing complete ✓ — no further log2 step is required")
        self.preprocessing_badge.setStyleSheet("font-weight:800;color:#176b35;")
        self.assess_btn.setEnabled(True)
        self.set_progress(100, "Log2/preprocessing step completed")
        self._set_busy(False)
        self.tabs.setCurrentWidget(self.batch_tab)
        self.log_message("Expression-scale processing completed. Ready to assess integration and batch effects.")

    # -------------------------- Batch assessment --------------------------
    def start_assessment(self):
        if not self.preprocessing_ready:
            QMessageBox.warning(self, "Preprocessing required", "Apply the selected log2 transformations first.")
            return
        self.assessment_result = None
        self.analysis_result = None
        self.figure_store = {}
        self.figure_list.clear()
        self.log_message("Starting common-gene integration and pre-correction batch-effect assessment...")
        self.set_progress(0, "Assessing batch-associated structure...")
        self._set_busy(True)

        thread = QThread()
        worker = AssessmentWorker(self.datasets)
        thread._worker = worker  # keep Python reference alive until the QThread finishes
        worker.moveToThread(thread)
        thread.started.connect(worker.run)
        worker.progress.connect(self.set_progress)
        worker.log.connect(self.log_message)
        worker.finished.connect(self._assessment_finished)
        worker.failed.connect(self._worker_failed)
        worker.finished.connect(thread.quit)
        worker.failed.connect(thread.quit)
        thread.finished.connect(worker.deleteLater)
        thread.finished.connect(thread.deleteLater)
        self._track_thread(thread)
        thread.start()

    def _assessment_finished(self, result):
        self.assessment_result = result
        self.analysis_result = None
        self.common_genes_value.value_label.setText(f"{result['merged'].shape[0]:,}")
        self.samples_value.value_label.setText(f"{result['merged'].shape[1]:,}")
        self.silhouette_value.value_label.setText(f"{result['qc_before']['silhouette']:.3f}")
        self.ratio_value.value_label.setText(f"{result['qc_before']['distance_ratio']:.3f}")

        level = result["guidance_level"]
        if level == "strong":
            style = "padding:10px;border-radius:7px;background:#fde8e8;font-weight:800;color:#9b1c1c;"
        elif level == "moderate":
            style = "padding:10px;border-radius:7px;background:#fff2cc;font-weight:800;color:#8a5a00;"
        elif level == "low":
            style = "padding:10px;border-radius:7px;background:#e5f5e9;font-weight:800;color:#176b35;"
        else:
            style = "padding:10px;border-radius:7px;background:#eef3f8;font-weight:800;color:#476176;"
        self.batch_guidance.setText("Batch-correction guidance: " + result["guidance_text"] + ". Review PCA and study design before deciding.")
        self.batch_guidance.setStyleSheet(style)
        self._populate_design_table(result["batch_group_table"])
        self.combat_btn.setEnabled(self.method_combo.currentIndex() == 0)
        self.save_btn.setEnabled(True)
        self.create_pre_figures()
        self.set_progress(100, "Batch-effect assessment completed")
        self._set_busy(False)
        self.log_message("Pre-correction assessment completed. Review the guidance and figures before running ComBat.")

    def _populate_design_table(self, table):
        self.design_table.setRowCount(len(table))
        for r, gse in enumerate(table.index):
            control = int(table.loc[gse, "control"]) if "control" in table.columns else 0
            disease = int(table.loc[gse, "disease"]) if "disease" in table.columns else 0
            for c, val in enumerate([str(gse), str(control), str(disease)]):
                item = QTableWidgetItem(val)
                item.setTextAlignment(Qt.AlignCenter)
                self.design_table.setItem(r, c, item)

    def _method_changed(self):
        no_correction = self.method_combo.currentIndex() == 1
        self.preserve_group.setEnabled(not no_correction)
        self.par_prior.setEnabled(not no_correction)
        self.mean_only.setEnabled(not no_correction)
        self.combat_btn.setEnabled((not no_correction) and self.assessment_result is not None)
        if no_correction and self.assessment_result is not None:
            self.post_qc_label.setText("No correction selected — merged matrix can be exported from the current assessment")

    # -------------------------- ComBat --------------------------
    def start_combat(self):
        if self.assessment_result is None:
            QMessageBox.warning(self, "Assessment required", "Assess integration and batch effects before running ComBat.")
            return
        if self.method_combo.currentIndex() != 0:
            QMessageBox.information(self, "No correction selected", "Select ComBat (pyComBat) to run batch correction.")
            return

        self.log_message("Running ComBat batch correction...")
        self.set_progress(0, "Running pyComBat...")
        self._set_busy(True)

        thread = QThread()
        worker = CombatWorker(
            self.assessment_result,
            preserve_group=self.preserve_group.isChecked(),
            par_prior=self.par_prior.isChecked(),
            mean_only=self.mean_only.isChecked(),
        )
        thread._worker = worker  # keep Python reference alive until the QThread finishes
        worker.moveToThread(thread)
        thread.started.connect(worker.run)
        worker.progress.connect(self.set_progress)
        worker.log.connect(self.log_message)
        worker.finished.connect(self._combat_finished)
        worker.failed.connect(self._worker_failed)
        worker.finished.connect(thread.quit)
        worker.failed.connect(thread.quit)
        thread.finished.connect(worker.deleteLater)
        thread.finished.connect(thread.deleteLater)
        self._track_thread(thread)
        thread.start()

    def _combat_finished(self, result):
        self.analysis_result = result
        self.post_qc_label.setText(
            f"Post-correction QC: silhouette={result['metrics']['silhouette_after']:.3f}; "
            f"distance ratio={result['metrics']['distance_ratio_after']:.3f}. {result['post_guidance']}."
        )
        if result["post_level"] == "good":
            self.post_qc_label.setStyleSheet("font-weight:800;color:#176b35;")
        elif result["post_level"] == "warning":
            self.post_qc_label.setStyleSheet("font-weight:800;color:#9b1c1c;")
        else:
            self.post_qc_label.setStyleSheet("font-weight:800;color:#8a5a00;")
        self.create_all_figures()
        self.save_btn.setEnabled(True)
        self.set_progress(100, "ComBat correction completed successfully")
        self._set_busy(False)
        self.log_message("ComBat correction and post-correction QC completed successfully.")
        self.tabs.setCurrentWidget(self.figures_tab)

    # -------------------------- Errors --------------------------
    def _worker_failed(self, trace):
        self._set_busy(False)
        self.progress.setValue(0)
        self.status_label.setText("Operation stopped because of an error")
        self.log_message("ERROR:\n" + trace)
        short = trace.strip().splitlines()[-1] if trace.strip() else "Unknown error"
        QMessageBox.critical(
            self,
            "Analysis Error",
            short + "\n\nThe complete traceback is available in the Log & Export tab.",
        )
        self.tabs.setCurrentWidget(self.log_export_tab)

    # -------------------------- Figures --------------------------
    def create_pre_figures(self):
        self.figure_store = {
            "PCA — Before correction": self.make_pca_before_figure(),
            "Expression boxplot — Before correction": self.make_boxplot_before_figure(),
            "Pre-correction batch QC": self.make_pre_qc_figure(),
        }
        self._refresh_figure_list()

    def create_all_figures(self):
        self.figure_store = {
            "PCA — Before / After": self.make_pca_figure(),
            "Expression boxplots — Before / After": self.make_boxplot_figure(),
            "Silhouette score": self.make_silhouette_figure(),
            "Within / between distance ratio": self.make_distance_figure(),
            "QC metrics summary": self.make_metrics_summary_figure(),
        }
        self._refresh_figure_list()

    def _refresh_figure_list(self):
        self.figure_list.clear()
        for key in self.figure_store:
            self.figure_list.addItem(QListWidgetItem(key))
        if self.figure_list.count():
            self.figure_list.setCurrentRow(0)

    def _figure_selected(self, current, previous):
        if current is None:
            return
        key = current.text()
        if key in self.figure_store:
            self.plot_panel.show_figure(self.figure_store[key])

    def _figure_source(self):
        return self.analysis_result if self.analysis_result is not None else self.assessment_result

    def make_pca_before_figure(self):
        r = self.assessment_result
        colors = dataset_color_map(r["dataset_order"])
        meta = r["metadata"]
        qc = r["qc_before"]
        fig = Figure(figsize=(9.2, 5.5), dpi=100)
        ax = fig.add_subplot(111)
        coords = qc["pca_coords"]
        for gse in r["dataset_order"]:
            idx = np.where(meta["dataset"].to_numpy() == gse)[0]
            ax.scatter(coords[idx, 0], coords[idx, 1], s=42, alpha=0.78, label=gse,
                       color=colors[gse], edgecolors="white", linewidths=0.4)
        ax.set_title("Before ComBat Correction", fontweight="bold", fontsize=15)
        ax.set_xlabel(f"PC1 ({qc['pca_variance'][0]:.1f}%)", fontweight="bold", fontsize=13)
        ax.set_ylabel(f"PC2 ({qc['pca_variance'][1]:.1f}%)", fontweight="bold", fontsize=13)
        ax.grid(True, linestyle="--", alpha=0.22)
        ax.legend(frameon=True, ncol=1)
        style_axis_bold(ax)
        fig.suptitle("PCA Assessment of Dataset-Associated Structure", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return fig

    def make_boxplot_before_figure(self):
        r = self.assessment_result
        colors = dataset_color_map(r["dataset_order"])
        order = r["dataset_order"]
        fig = Figure(figsize=(9.2, 5.5), dpi=100)
        ax = fig.add_subplot(111)
        data = [r["before_distributions"][gse] for gse in order]
        bp = ax.boxplot(data, patch_artist=True, showfliers=False,
                        medianprops={"color": "black", "linewidth": 1.8},
                        boxprops={"linewidth": 1.2}, whiskerprops={"linewidth": 1.1}, capprops={"linewidth": 1.1})
        for patch, gse in zip(bp["boxes"], order):
            patch.set_facecolor(colors[gse]); patch.set_alpha(0.80)
        ax.set_xticks(np.arange(1, len(order) + 1)); ax.set_xticklabels(order, rotation=30, ha="right")
        ax.set_xlabel("GEO Dataset", fontweight="bold", fontsize=13)
        ax.set_ylabel("Expression Value", fontweight="bold", fontsize=13)
        ax.set_title("Before ComBat Correction", fontweight="bold", fontsize=15)
        ax.grid(True, axis="y", linestyle="--", alpha=0.22)
        style_axis_bold(ax)
        fig.suptitle("Expression Distribution by GEO Dataset", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return fig

    def make_pre_qc_figure(self):
        r = self.assessment_result
        qc = r["qc_before"]
        fig = Figure(figsize=(9.2, 5.5), dpi=100)
        ax1, ax2 = fig.subplots(1, 2)
        b1 = ax1.bar(["Before"], [qc["silhouette"]], color=["#d95f4f"], width=0.55)
        ax1.axhline(0, color="black", linewidth=1)
        ax1.set_ylim(-1, 1)
        ax1.set_ylabel("Mean Silhouette Score", fontweight="bold", fontsize=13)
        ax1.set_title("Dataset Silhouette", fontweight="bold", fontsize=15)
        ax1.text(b1[0].get_x()+b1[0].get_width()/2, qc["silhouette"] + (0.04 if qc["silhouette"] >= 0 else -0.07),
                 f"{qc['silhouette']:.3f}", ha="center", va="bottom" if qc["silhouette"] >= 0 else "top",
                 fontweight="bold", fontsize=12)
        style_axis_bold(ax1)
        b2 = ax2.bar(["Before"], [qc["distance_ratio"]], color=["#d95f4f"], width=0.55)
        ax2.axhline(1.0, color="#333333", linestyle="--", linewidth=1.4, label="Reference = 1")
        ax2.set_ylabel("Within / Between Distance Ratio", fontweight="bold", fontsize=13)
        ax2.set_title("Batch Distance Ratio", fontweight="bold", fontsize=15)
        ax2.text(b2[0].get_x()+b2[0].get_width()/2, qc["distance_ratio"] + 0.03,
                 f"{qc['distance_ratio']:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=12)
        ax2.legend(); style_axis_bold(ax2)
        for ax in [ax1, ax2]: ax.grid(True, axis="y", linestyle="--", alpha=0.22)
        fig.suptitle("Pre-correction Batch-QC Metrics", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return fig

    def make_pca_figure(self):
        r = self.analysis_result
        colors = dataset_color_map(r["dataset_order"])
        meta = r["metadata"]
        fig = Figure(figsize=(12.2, 5.4), dpi=100)
        ax1, ax2 = fig.subplots(1, 2)
        for ax, qc, title in [(ax1, r["qc_before"], "Before ComBat Correction"), (ax2, r["qc_after"], "After ComBat Correction")]:
            coords = qc["pca_coords"]
            for gse in r["dataset_order"]:
                idx = np.where(meta["dataset"].to_numpy() == gse)[0]
                ax.scatter(coords[idx, 0], coords[idx, 1], s=40, alpha=0.76, label=gse,
                           color=colors[gse], edgecolors="white", linewidths=0.4)
            ax.set_title(title, fontweight="bold", fontsize=15)
            ax.set_xlabel(f"PC1 ({qc['pca_variance'][0]:.1f}%)", fontweight="bold", fontsize=13)
            ax.set_ylabel(f"PC2 ({qc['pca_variance'][1]:.1f}%)", fontweight="bold", fontsize=13)
            ax.grid(True, linestyle="--", alpha=0.22)
            ax.legend(title=None, frameon=True, loc="best")
            style_axis_bold(ax)
        fig.suptitle("PCA Assessment of Dataset-Associated Structure", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return fig

    def make_boxplot_figure(self):
        r = self.analysis_result
        colors = dataset_color_map(r["dataset_order"])
        order = r["dataset_order"]
        fig = Figure(figsize=(12.2, 5.4), dpi=100)
        ax1, ax2 = fig.subplots(1, 2)
        for ax, dist, title in [(ax1, r["before_distributions"], "Before ComBat Correction"), (ax2, r["after_distributions"], "After ComBat Correction")]:
            data = [dist[gse] for gse in order]
            bp = ax.boxplot(data, patch_artist=True, showfliers=False,
                            medianprops={"color": "black", "linewidth": 1.8},
                            boxprops={"linewidth": 1.2}, whiskerprops={"linewidth": 1.1}, capprops={"linewidth": 1.1})
            for patch, gse in zip(bp["boxes"], order):
                patch.set_facecolor(colors[gse]); patch.set_alpha(0.80)
            ax.set_xticks(np.arange(1, len(order) + 1)); ax.set_xticklabels(order, rotation=30, ha="right")
            ax.set_title(title, fontweight="bold", fontsize=15)
            ax.set_xlabel("GEO Dataset", fontweight="bold", fontsize=13)
            ax.set_ylabel("Expression Value", fontweight="bold", fontsize=13)
            ax.grid(True, axis="y", linestyle="--", alpha=0.22)
            style_axis_bold(ax)
        fig.suptitle("Expression Distribution by GEO Dataset", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return fig

    def make_silhouette_figure(self):
        r = self.analysis_result
        before, after = r["qc_before"], r["qc_after"]
        scores = [before["silhouette"], after["silhouette"]]
        fig = Figure(figsize=(11.6, 5.3), dpi=100)
        ax1, ax2 = fig.subplots(1, 2)
        bars = ax1.bar(["Before", "After"], scores, color=["#d95f4f", "#2f7ed8"], alpha=0.86)
        ax1.axhline(0, color="black", linewidth=1)
        ax1.set_ylim(-1, 1)
        ax1.set_ylabel("Mean Silhouette Score", fontweight="bold", fontsize=13)
        ax1.set_title("Dataset-Associated Clustering", fontweight="bold", fontsize=15)
        for bar, score in zip(bars, scores):
            y = score + (0.035 if score >= 0 else -0.075)
            ax1.text(bar.get_x()+bar.get_width()/2, y, f"{score:.3f}", ha="center",
                     va="bottom" if score >= 0 else "top", fontweight="bold", fontsize=12)
        ax1.grid(True, axis="y", linestyle="--", alpha=0.22); style_axis_bold(ax1)

        vp = ax2.violinplot([before["silhouette_samples"], after["silhouette_samples"]], showmeans=True, showmedians=True)
        for body, color in zip(vp["bodies"], ["#d95f4f", "#2f7ed8"]):
            body.set_facecolor(color); body.set_alpha(0.75)
        for key in ["cbars", "cmins", "cmaxes", "cmeans", "cmedians"]:
            if key in vp:
                vp[key].set_color("black"); vp[key].set_linewidth(1.2)
        ax2.set_xticks([1, 2]); ax2.set_xticklabels(["Before", "After"])
        ax2.set_ylim(-1, 1)
        ax2.set_ylabel("Sample Silhouette Score", fontweight="bold", fontsize=13)
        ax2.set_title("Sample-Level Distribution", fontweight="bold", fontsize=15)
        ax2.grid(True, axis="y", linestyle="--", alpha=0.22); style_axis_bold(ax2)
        fig.suptitle("Silhouette Analysis of Dataset-Origin Clustering", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return fig

    def make_distance_figure(self):
        m = self.analysis_result["metrics"]
        ratio = [m["distance_ratio_before"], m["distance_ratio_after"]]
        within = [m["mean_within_before"], m["mean_within_after"]]
        between = [m["mean_between_before"], m["mean_between_after"]]
        fig = Figure(figsize=(11.6, 5.3), dpi=100)
        ax1, ax2 = fig.subplots(1, 2)
        bars = ax1.bar(["Before", "After"], ratio, color=["#d95f4f", "#2f7ed8"], alpha=0.86)
        ax1.axhline(1.0, color="#333333", linestyle="--", linewidth=1.4, label="Reference = 1")
        ax1.set_ylabel("Within / Between Distance Ratio", fontweight="bold", fontsize=13)
        ax1.set_title("Within-to-Between Batch Distance Ratio", fontweight="bold", fontsize=15)
        for bar, value in zip(bars, ratio):
            ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f"{value:.3f}",
                     ha="center", va="bottom", fontweight="bold", fontsize=12)
        ax1.legend(); ax1.grid(True, axis="y", linestyle="--", alpha=0.22); style_axis_bold(ax1)

        x = np.arange(2); width = 0.36
        b1 = ax2.bar(x-width/2, within, width, label="Mean within-batch distance", color="#4daf4a", alpha=0.86)
        b2 = ax2.bar(x+width/2, between, width, label="Mean between-batch distance", color="#984ea3", alpha=0.86)
        ax2.set_xticks(x); ax2.set_xticklabels(["Before", "After"])
        ax2.set_ylabel("Mean Standardized Euclidean Distance", fontweight="bold", fontsize=13)
        ax2.set_title("Within- vs Between-Batch Distances", fontweight="bold", fontsize=15)
        for bs in [b1, b2]:
            for bar in bs:
                ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{bar.get_height():.2f}",
                         ha="center", va="bottom", fontweight="bold", fontsize=10)
        ax2.legend(); ax2.grid(True, axis="y", linestyle="--", alpha=0.22); style_axis_bold(ax2)
        fig.suptitle("Batch Distance Analysis", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        return fig

    def make_metrics_summary_figure(self):
        m = self.analysis_result["metrics"]
        fig = Figure(figsize=(10.6, 7.4), dpi=100)
        axes = fig.subplots(2, 2)
        ax1, ax2, ax3, ax4 = axes.ravel()
        svals = [m["silhouette_before"], m["silhouette_after"]]
        bars = ax1.bar(["Before", "After"], svals, color=["#d95f4f", "#2f7ed8"])
        ax1.axhline(0, color="black", linewidth=1); ax1.set_ylim(-1, 1)
        ax1.set_ylabel("Silhouette Score", fontweight="bold"); ax1.set_title("Dataset Silhouette", fontweight="bold")
        for b, v in zip(bars, svals):
            ax1.text(b.get_x()+b.get_width()/2, v+(0.03 if v>=0 else -0.07), f"{v:.3f}",
                     ha="center", va="bottom" if v>=0 else "top", fontweight="bold")
        style_axis_bold(ax1)

        rvals = [m["distance_ratio_before"], m["distance_ratio_after"]]
        bars = ax2.bar(["Before", "After"], rvals, color=["#d95f4f", "#2f7ed8"])
        ax2.axhline(1.0, color="black", linestyle="--", linewidth=1.2)
        ax2.set_ylabel("Within / Between Ratio", fontweight="bold"); ax2.set_title("Batch Distance Ratio", fontweight="bold")
        for b, v in zip(bars, rvals):
            ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f"{v:.3f}", ha="center", va="bottom", fontweight="bold")
        style_axis_bold(ax2)

        x = np.arange(2); width = 0.36
        ax3.bar(x-width/2, [m["mean_within_before"], m["mean_within_after"]], width, label="Within", color="#4daf4a")
        ax3.bar(x+width/2, [m["mean_between_before"], m["mean_between_after"]], width, label="Between", color="#984ea3")
        ax3.set_xticks(x); ax3.set_xticklabels(["Before", "After"])
        ax3.set_ylabel("Mean Distance", fontweight="bold"); ax3.set_title("Mean Batch Distances", fontweight="bold"); ax3.legend(); style_axis_bold(ax3)

        silhouette_to_zero = abs(m["silhouette_before"]) - abs(m["silhouette_after"])
        ratio_to_one = abs(m["distance_ratio_before"] - 1) - abs(m["distance_ratio_after"] - 1)
        vals = [silhouette_to_zero, ratio_to_one]
        bars = ax4.bar(["Silhouette\ntoward 0", "Distance ratio\ntoward 1"], vals,
                       color=["#3c8dbc" if v >= 0 else "#c94c4c" for v in vals])
        ax4.axhline(0, color="black", linewidth=1)
        ax4.set_ylabel("Directional Improvement", fontweight="bold"); ax4.set_title("QC Directional Change", fontweight="bold")
        for b, v in zip(bars, vals):
            ax4.text(b.get_x()+b.get_width()/2, v+(0.01 if v>=0 else -0.02), f"{v:+.3f}",
                     ha="center", va="bottom" if v>=0 else "top", fontweight="bold")
        style_axis_bold(ax4)
        for ax in axes.ravel(): ax.grid(True, axis="y", linestyle="--", alpha=0.20)
        fig.suptitle("Batch-Correction QC Metrics Summary", fontweight="bold", fontsize=17)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        return fig

    # -------------------------- Export --------------------------
    def choose_output_directory(self):
        directory = QFileDialog.getExistingDirectory(self, "Choose output folder")
        if directory:
            self.output_directory = Path(directory)
            self.output_path.setText(str(self.output_directory))

    def save_all_results(self):
        base_result = self.analysis_result if self.analysis_result is not None else self.assessment_result
        if base_result is None:
            QMessageBox.information(self, "No results", "Run the integration/batch-effect assessment first.")
            return
        if self.output_directory is None:
            self.choose_output_directory()
            if self.output_directory is None:
                return

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        project = self.output_directory / f"GeneExpression_BatchCorrection_{stamp}"
        data_dir = project / "data"
        qc_dir = project / "qc"
        fig_dir = project / "figures"
        report_dir = project / "reports"
        for d in [data_dir, qc_dir, fig_dir, report_dir]:
            d.mkdir(parents=True, exist_ok=True)

        try:
            self.set_progress(5, "Saving merged expression matrix...")
            QApplication.processEvents()
            base_result["merged"].to_csv(data_dir / "merged_preprocessed_expression.csv")
            if self.analysis_result is not None:
                self.analysis_result["corrected"].to_csv(data_dir / "combat_corrected_expression.csv")

            self.set_progress(20, "Saving metadata and QC tables...")
            QApplication.processEvents()
            base_result["metadata"].to_csv(data_dir / "sample_metadata.csv", index=False)
            base_result["dataset_summary"].to_csv(qc_dir / "dataset_summary.csv", index=False)
            base_result["batch_group_table"].to_csv(qc_dir / "batch_by_biological_group.csv")

            if self.analysis_result is not None:
                m = self.analysis_result["metrics"]
                pd.DataFrame([m]).to_csv(qc_dir / "batch_qc_metrics.csv", index=False)
                settings = self.analysis_result["analysis_settings"]
            else:
                q = base_result["qc_before"]
                pd.DataFrame([{
                    "silhouette_before": q["silhouette"],
                    "distance_ratio_before": q["distance_ratio"],
                    "mean_within_before": q["mean_within"],
                    "mean_between_before": q["mean_between"],
                    "batch_guidance": base_result["guidance_text"],
                }]).to_csv(qc_dir / "precorrection_batch_qc_metrics.csv", index=False)
                settings = {
                    "app_name": APP_NAME,
                    "app_version": APP_VERSION,
                    "created_at": datetime.now().isoformat(timespec="seconds"),
                    "batch_correction_run": False,
                    "common_genes_retained": int(base_result["merged"].shape[0]),
                    "total_samples": int(base_result["merged"].shape[1]),
                }

            self.set_progress(35, "Saving analysis settings and log...")
            QApplication.processEvents()
            with open(report_dir / "analysis_settings.json", "w", encoding="utf-8") as f:
                json.dump(settings, f, indent=2, default=str)
            with open(report_dir / "analysis_log.txt", "w", encoding="utf-8") as f:
                f.write(self.log_text.toPlainText())
            with open(report_dir / "USER_MANUAL.html", "w", encoding="utf-8") as f:
                f.write(USER_MANUAL_HTML)

            figures = self.figure_store
            if figures:
                total = len(figures) * 4
                done = 0
                for title, fig in figures.items():
                    safe = re.sub(r"[^A-Za-z0-9_-]+", "_", title).strip("_")
                    for ext, dpi in [("png", 600), ("tiff", 600), ("svg", None), ("pdf", None)]:
                        done += 1
                        self.set_progress(40 + int(done / total * 55), f"Exporting {safe}.{ext}...")
                        QApplication.processEvents()
                        kwargs = {"bbox_inches": "tight", "facecolor": "white"}
                        if dpi is not None:
                            kwargs["dpi"] = dpi
                        fig.savefig(fig_dir / f"{safe}.{ext}", **kwargs)

            self.set_progress(100, "All results saved successfully")
            self.log_message(f"All results saved to: {project}")
            QMessageBox.information(self, "Export complete", f"All results were saved to:\n\n{project}")
        except Exception:
            trace = traceback.format_exc()
            self.log_message("EXPORT ERROR:\n" + trace)
            self.set_progress(0, "Export failed")
            QMessageBox.critical(self, "Export Error", trace.strip().splitlines()[-1])

    # -------------------------- Help / About --------------------------
    def show_user_manual(self):
        TextDialog("User Manual", USER_MANUAL_HTML, self, 930, 720).exec_()

    def show_about(self):
        html = f"""
        <h2>{APP_NAME}</h2>
        <p><b>Version:</b> {APP_VERSION}</p>
        <p>Guided processed-microarray integration and batch-effect assessment using pyComBat.</p>
        <p><b>Developed by:</b><br>{DEVELOPERS}</p>
        <p><b>Affiliation:</b><br>{AFFILIATION}</p>
        <p><b>Contact:</b> {CONTACT_EMAIL}</p>
        <hr>
        <p><b>Key outputs:</b> integrated common-gene matrix, ComBat-corrected matrix, PCA,
        expression boxplots, dataset-origin silhouette analysis, within-to-between batch-distance analysis,
        QC tables, analysis settings, and publication figures.</p>
        """
        TextDialog("About", html, self, 650, 480).exec_()

# ============================================================
# 6. LAUNCH
# ============================================================
def launch_gene_expression_batch_studio():
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    window = GeneExpressionBatchStudio()
    window.show()
    app.exec_()
    return window


if __name__ == "__main__":
    _gene_batch_window = launch_gene_expression_batch_studio()
